# 05 — Model family & feature set (team 07)

**Owner:** Patrick, filled in **after** 01–04 are merged into `team-07-main`, from their findings tables.
*(Filled in by Kingston for Deliverable 4.)*

This is the actual deliverable: a named model family, a feature set produced by code, and a reason
for every feature kept or dropped. The final model, [`../xgboost_model.py`](../xgboost_model.py), uses exactly this feature code: this notebook imports it
from the script, so the two can't drift apart.

Choose the model in the D2 order: **data shape → loss → what AA needs → cost → only then score.**

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

In [ ]:
%run ./00_load_and_preprocess.ipynb
# gives you: pax, airports, TARGET, REPO_ROOT, RESULTS_DIR

## 1. Model family

**Chosen family:** gradient-boosted trees (XGBoost), trained as one model across all routes, with its route forecasts
scaled each month to add up to a seasonal-naive network total.

We chose XGBoost, a gradient-boosted tree model, trained as one model across all routes. Each route is its own monthly time series, but there are about 17,000 of them with only 91 months each, so fitting a separate ARIMA model per route would be slow and each one would see very little data. One shared tree model can learn patterns that repeat across routes, like summer peaks or small routes being more jumpy, and trees handle the wide range of route sizes, the airport codes, and missing months without extra work.

The other choice that shapes everything is how we set up the forecast. Deliverable 5 asks for a forecast of two years ahead as a whole, not just next month. So every row is built from a forecast date: the model may only use data up to that date, and it predicts each of the next 1 to 24 months, with the number of months ahead (h) as a feature. The test is one real forecast made at July 2024 for August 2024 through July 2026. Training uses forecast dates from July 2022 to June 2024, because before mid-2022 the history the features look back on is mostly COVID.

| EDA observation (from notebook #) | Points toward / away from |
|---|---|
| One row is one route in one month, about 17,000 routes, 91 months (01) | Toward one global model across all routes. Away from fitting 17,000 separate ARIMA models |
| Strong yearly seasonality, summer peaks and winter lows (03) | Toward month features and same-month-last-year features. Any model works if given these |
| Passengers range from under 1 to over 100,000 per route-month, heavy right tail (02, 03) | Toward training on log(1 + passengers). Trees handle the wide range well |
| COVID collapse in 2020 and recovery through 2022 (01, 03) | Away from models that need long clean history like LSTM. Toward training only on forecast dates after mid-2022 |
| 150 airports, 17,119 routes, airport table joins cleanly (04) | Toward trees, which can use airport codes as categories without one-hot blowing up |
| OD_REV_USD is known only after the month happens (03, 04) | Never use same-month revenue. Only past revenue (yield) is allowed |
| D5 needs a forecast two years ahead as a whole (DELIVERABLES.md) | Toward a forecast-date setup where every feature comes from before the forecast date |
| 13.6% of route-months are high outliers, about two thirds on the top 10% of routes (02) | Keep them. Weight training by route size and judge with WAPE |
| Training years 2022 to 2024 are a recovery period (03) | Trees can't extrapolate, so the network total comes from seasonal naive and XGBoost only splits it across routes |

## 2. Build the feature set

The feature code and the shared settings live in `../xgboost_model.py` and are imported here, so this
notebook and the final model always use the same features, forecast dates, test window, and metrics.

In [ ]:
import sys

sys.path.insert(0, str(MODELING_DIR))
from xgboost_model import (
    FEATURES, FIRST_TRAIN_ORIGIN, HORIZON, TEST_ORIGIN, VAL_START,
    build_features, naive_forecast, reconcile, route_weight, score,
)

print("forecast horizon:", HORIZON, "months | test forecast date:", f"{TEST_ORIGIN:%Y-%m}",
      "| first training forecast date:", f"{FIRST_TRAIN_ORIGIN:%Y-%m}", "| features:", len(FEATURES))

In [ ]:
train_origins = pd.date_range(FIRST_TRAIN_ORIGIN, TEST_ORIGIN - pd.DateOffset(months=1), freq="MS")
train_set = build_features(pax, airports, train_origins, max_target=TEST_ORIGIN)
test_set = build_features(pax, airports, [TEST_ORIGIN])

X, y = train_set[FEATURES], train_set[TARGET]
X_test, y_test = test_set[FEATURES], test_set[TARGET]
print(f"train: {len(train_set):,} rows from {len(train_origins)} forecast dates, "
      f"targets {train_set['DATE'].min():%Y-%m} to {train_set['DATE'].max():%Y-%m}")
print(f"test:  {len(test_set):,} rows, one forecast made at {TEST_ORIGIN:%Y-%m} for "
      f"{test_set['DATE'].min():%Y-%m} to {test_set['DATE'].max():%Y-%m}")

### Shared metrics and baseline

`score()` gives WAPE (total absolute error divided by total passengers, the headline number because it
weights big routes more, like the business does), MAE, RMSE, and bias. `naive_forecast()` is the seasonal
naive baseline: the same month in the latest year seen.

In [ ]:
pd.DataFrame({"seasonal naive": score(y_test, naive_forecast(test_set))}).T.round(3)

## 3. Rationale per feature

| Feature | Kept / engineered / dropped | Reason (cite the EDA notebook) |
|---|---|---|
| `h` | Engineered | Months between the forecast date and the target month. Lets one model learn that far-out months are less certain (D5 two-year horizon) |
| `month_sin`, `month_cos` | Engineered | Target month as a circle so December sits next to January (03) |
| `last_same_month` | Engineered | Same month in the latest year we've seen. Captures each route's seasonality and size (03). It is also the naive baseline |
| `level_12`, `level_3` | Engineered | Route's average over the last 12 and last 3 months before the forecast date, its current size (01, 03) |
| `same_vs_level12`, `level3_vs_level12` | Engineered | How peaky the target month is for this route, and whether the route is trending up or down lately (03) |
| `growth_12` | Engineered | Last 12 months vs the 12 before, the route's yearly growth (03) |
| `pax_2019_same_month`, `level12_vs_2019` | Engineered | Pre-COVID level of the route, and how far it has recovered. 2019 is always before any forecast date (01, 03) |
| `yield_12` | Engineered | Average revenue per passenger over the last 12 months. Past revenue is allowed, same-month revenue is not (03, 04) |
| `org_level_12`, `des_level_12` | Engineered | Total traffic at each airport, a hub-size signal for small routes (04) |
| `months_seen_12` | Engineered | How many of the last 12 months the route reported. Routes with gaps behave differently (01) |
| `distance_mi` | Engineered | Great-circle distance from the airport latitudes and longitudes (04) |
| `domestic` | Engineered | Both airports in the US (04) |
| `ORG_AIRPRT`, `DES_AIRPRT` | Kept | 150 airports each, used as native categories in XGBoost instead of one-hot (04) |
| `org_/des_runway_ft`, `org_/des_world_area`, `org_/des_country` | Engineered | Airport attributes joined to each end of the route (04). Improved both validation and test |
| `covid_months_in_history` | Engineered | How many of the 12 months before the forecast date were COVID months (03's regime flag) |
| `route_hist_mean`, `route_hist_count`, `route_month_hist_mean` | Engineered | Route's all-time average, months reported, and all-time same-month average up to the forecast date (teammate's `XGBoost_model.ipynb`). Improved both validation and test |
| `DEP_YEAR` | Dropped | Trees can't extrapolate a year number into 2025 and 2026, and the trend is carried by the level features instead (03) |
| `OD_REV_USD` (same month) | Dropped | Leakage, only known after the month happens (03, 04) |
| `Unnamed: 0`, `SF_LOAD_TMS` | Dropped | Row counter and a single load timestamp (01) |
| `STATE_PROVNC_CD`, airport names | Dropped | 52% missing, and names add nothing beyond the code (01, 04) |
| Route-size training weight (not a feature) | Engineered | Square root of the route's 12-month average, so big routes count more, matching WAPE (02) |
| Route as one category (17,119 values) | Dropped | Too many levels. Origin plus destination plus the route's own history carry the same information (04) |

Every feature above is computed only from months up to the forecast date, so none of them can leak the answer. The model is trained on log(1 + passengers) because route sizes range from under 1 to over 100,000 passengers a month, and the test forecasts are turned back into passengers before scoring. Missing route-months are left out of scoring and show up as gaps in the features, which XGBoost handles on its own.

The model itself (training, reconciliation to the network total, prediction ranges, tests, and the real two-year forecast) is in ../xgboost_model.py, and the results are in ../results/model-comparison.md.